
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Building Vector Search for Retrieval

## Overview
Welcome! In this demo, we’ll explore how to build a vector search solution for document retrieval using Databricks. We’ll walk through transforming document chunks into semantic vectors, creating a vector search index, and leveraging advanced search and re-ranking techniques to boost retrieval precision.

In real-world retrieval scenarios, converting unstructured documents into searchable semantic vectors enables more accurate and context-aware results. This workflow empowers you to efficiently find relevant information, even when keywords don’t match exactly.

## Learning Objectives
- **Identify** the steps to compute document embeddings using the GTE model from Foundation Model APIs.
- **Configure** and create a vector search index using both SDK and UI methods.
- **Implement** and compare query, hybrid, and full-text search methods, including filtering by document path.
- **Improve** search precision through re-ranking and understand its impact on retrieval quality.
- **Apply** best practices for balancing computational cost, accuracy, and index refresh strategies.

## Requirements
- A pre-created **vector search endpoint**. This is pre-created for you.
- **Serverless Compute (environment version 4)**. Follow the instructions [here](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) to select the appropriate environment version.
- Required libraries are added to **Dependencies** of Serverless compute configuration.
- Access to Foundation Model APIs for embedding generation.
- Appropriate permissions to create and manage vector search indexes.

## Setup

Run the code below to install required libraries and configure your classroom environment. This step ensures all dependencies are available and your workspace is ready for the demo.


In [0]:
%run ../Includes/Classroom-Setup-03

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


**Set default catalog and schema:**

Default catalog and schema are set.


**Prepare Datasets:**

✅ Volume orion_docs already exists. Skipping copy.
✅ Volume orion_text already exists. Skipping copy.


**Check for Serverless**

✅ Environment check passed: Serverless 4 detected.


**Check If Vector Search Endpoint is Ready**

✅ Vector Search endpoint is ready. 

✍️ Vector Search endpoint to use: vs_endpoint_1


## A. Prepare Source Table

Vector Search requires the source table to have Change Data Feed (CDF) enabled. If the table already has this feature enabled, we don't need to make any changes. If not, we can enable it as follows.

Also, note that we need to have a unique ID for the table that will be required when creating the vector search index.

In [0]:
# Enable Change Data Feed for vector search synchronization
spark.sql(f"ALTER TABLE {docs_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Display sample data to understand table structure
display(spark.sql(f"SELECT * FROM {docs_table} LIMIT 5"))

path chunk id dbfs:/Volumes/dbacademy/labuser13859245_1773140998/orion_docs/01_Orion_A1_Product_Overview.pdf Orion Robotics

Orion A1 Product Overview
© 2025 Orion Robotics – Confidential
The Orion A1 humanoid service robot is the flagship model in the Orion Robotics product line. Designed for adaptive environments, A1 combines modular hardware, precision motion control, and deep learning-based sensory perception to perform tasks ranging from hospitality assistance to warehouse logistics.
Since its initial release in 2023, Orion A1 has been deployed in over 40 countries. Each production cycle incorporates firmware refinements and mechanical upgrades based on telemetry data collected from field units. The 2025 A1 model introduces an improved gait controller and battery module rated for 10 hours of continuous operation. 0 dbfs:/Volumes/dbacademy/labuser13859245_1773140998/orion_docs/01_Orion_A1_Product_Overview.pdf == page ==

1. System Architecture

The Orion A1 platform is built on a modular architecture comprising four key subsystems: motion, vision, cognition, and power management. The motion subsystem employs brushless torque-controlled actuators for high-precision joint control. The vision module includes dual stereo cameras calibrated for depth perception and object tracking under dynamic lighting.
Global Shipments by Variant (2024)
Units Shipped
0
1000
A1
A2
A3
Model

Figure 1. Global shipments by product variant (2024).

2. Product Specifications

 Attribute Specification Height 1.55 m Weight 42 kg Battery Life 10 hours continuous operation Charging Time 90 minutes (fast charge) Processor OrionCore X7 Neural SoC Operating Temperature 0–45 °C 

The A1 platform supports over-the-air (OTA) firmware updates, allowing customers to seamlessly deploy safety patches and performance enhancements. Each robot maintains an encrypted event log for diagnostic analysis and predictive maintenance.
For further integration, Orion provides a RESTful API and ROS-compatible SDK. This enables developers to design custom behaviors, sensor fusion algorithms, and environment-specific routines using Python or C++
End of Document 1 dbfs:/Volumes/dbacademy/labuser13859245_1773140998/orion_docs/02_Orion_Battery_Management_System_Manual.pdf Orion Robotics

Orion A1 - Battery Management System Manual
© 2025 Orion Robotics – Internal Engineering Reference
This manual provides an in-depth overview of the Orion A1 Battery Management System (BMS), covering hardware design, safety mechanisms, and maintenance best practices. The BMS is a critical subsystem responsible for monitoring, protecting, and optimizing the lithium-ion battery modules powering the Orion A1 platform.
The Orion BMS ensures long-term battery health by dynamically balancing cell voltages, regulating current flow, and monitoring thermal conditions through embedded temperature sensors. The system's adaptive charge algorithm was developed in collaboration with the Orion Power Systems division, achieving a balance between performance and safety.

== page ==

1. System Architecture

The BMS consists of two primary components: the Master Control Unit (MCU) and distributed Cell Monitoring Boards (CMBs). The MCU consolidates telemetry from each CMB, computes charge/discharge limits, and communicates with the motion controller via the OrionBus CAN protocol. Each CMB continuously tracks cell voltage (±1mV accuracy), temperature, and impedance.
Thermal management is handled by a closed-loop liquid cooling system, which is activated automatically when the cell temperature exceeds 38°C. The system's safety layer includes dual overcurrent protection and hardware-based short-circuit isolation. If a thermal runaway condition is detected, the BMS triggers a controlled shutdown sequence within 200 milliseconds.
Battery Efficiency vs Temperature
Efficiency (%)
95.0
92.5
90.0
87.5
0
10
20
30
40
50
Temperature (°C)

Figure 1. Efficiency of Orion A1 battery pack across temperature ranges. 2 dbfs:/Volumes/dba

## B. Computing Embeddings for Document Chunks

Embeddings are high-dimensional vector representations of text that capture semantic meaning, enabling powerful similarity search and retrieval. In Databricks, embeddings allow us to find contextually relevant document chunks, even when keywords do not match exactly.

Databricks supports two main approaches for generating embeddings:
* **Managed embeddings:** Vector Search automatically computes and manages embeddings for you, simplifying setup and maintenance. This is the recommended approach for most use cases.
* **Manual embeddings:** You can generate embeddings externally (using MLflow Deployments, Hugging Face, OpenAI, etc.) and store them in a column. For large datasets, you can use a Spark UDF to compute embeddings for each row in a text column.

In this demo, we will use **managed embeddings**, allowing Vector Search to compute and maintain the embeddings for us. This approach streamlines the workflow and ensures compatibility with Databricks' vector search features.

In [0]:
import mlflow.deployments

# Initialize deployment client for accessing embedding models
deploy_client = mlflow.deployments.get_deploy_client("databricks")

# Generate embeddings for a sample question
question = "How Generative AI impacts humans?"
response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [question]})
embeddings = [e["embedding"] for e in response.data]

# Display embedding information
print("Embedding for question:", embeddings[0])
print("Embedding shape:", len(embeddings[0]))

Embedding for question: [1.201171875, -0.02044677734375, -0.4375, 0.156494140625, -0.250732421875, -0.50927734375, 0.96044921875, -1.4052734375, -1.638671875, 0.1719970703125, -0.2384033203125, -0.1661376953125, 0.69921875, -0.2437744140625, -0.6474609375, -0.35986328125, 0.88134765625, -0.69189453125, 0.485107421875, -0.9775390625, -0.12396240234375, -1.1875, 0.4208984375, -0.701171875, 1.2626953125, -0.371826171875, -0.2374267578125, -0.59619140625, 0.496337890625, -0.1798095703125, 1.1171875, 0.423095703125, -0.72509765625, 0.51953125, 0.281982421875, 0.19091796875, 0.7890625, -0.235107421875, -0.41259765625, 0.78857421875, 0.5380859375, 0.058746337890625, -0.2169189453125, -0.49853515625, -0.487548828125, -0.254150390625, 0.51123046875, -1.076171875, -0.57666015625, 0.3505859375, -0.2412109375, -0.6044921875, 0.48046875, -0.2122802734375, -0.41455078125, -0.428466796875, 0.6171875, -0.499267578125, -0.5263671875, 0.3330078125, -0.164794921875, -0.157958984375, 0.370361328125, -0.15

**💡 Question:** What does the `1024` embedding shape mean? Can this be changed for the embedding model we're using?

## C. Creating a Vector Search Index

Now that we have embeddings, we’ll create a vector search index to enable fast and accurate retrieval. Databricks supports two main approaches:

- **SDK Method:** Use the databricks-vectorsearch SDK to create an index with computed embeddings.
- **UI Method:** Use the Databricks UI to create an index with managed or computed embeddings.

We’ll use the pre-created vector search endpoint defined in the setup section. For instructions on creating an endpoint, refer to the [endpoint documentation](https://docs.databricks.com/aws/en/vector-search/create-vector-search#create-a-vector-search-endpoint-using-the-ui).


### C1. Creating Index via SDK

In this step, we'll create a vector search index using the Databricks SDK. We'll use managed embeddings, allowing Databricks to automatically compute and maintain vector representations for each chunk.

For more details, see the [Vector Search SDK documentation](https://api-docs.databricks.com/python/vector-search/index.html).

**Note:** Index refresh modes can be set to manual or sync, depending on your update requirements.

In [0]:
from databricks.vector_search.client import VectorSearchClient

# Initialize the Vector Search client
vsc = VectorSearchClient(disable_notice=True)

# Define index name using three-level naming convention
index_name = f"{catalog}.{schema}.docs_chunked_index"

# Create the index using managed embeddings with Delta Sync
vsc.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint,
    index_name=index_name,
    source_table_name=docs_table,
    primary_key='id',
    embedding_source_column="chunk",
    embedding_model_endpoint_name="databricks-gte-large-en",
    pipeline_type="TRIGGERED",
)
print(f"Index '{index_name}' created for table '{docs_table}' using endpoint '{vector_search_endpoint}'.")

Index 'dbacademy.labuser13859245_1773140998.docs_chunked_index' created for table 'dbacademy.labuser13859245_1773140998.docs_chunked' using endpoint 'vs_endpoint_1'.


### C1. Creating Index via UI (Optional)

We can also create a vector search index using the Databricks UI, which supports both managed and computed embeddings. This method is recommended for users who prefer a graphical interface or want to leverage managed embedding workflows.

For step-by-step instructions, see the [Vector Search UI documentation](https://docs.databricks.com/aws/en/vector-search/create-vector-search#create-index-using-the-ui).

1. In the left sidebar, click **Catalog** to open Catalog Explorer.
2. Find and select your Delta table.
3. Click **Create** (top right) and choose **Vector search index**.
4. In the dialog, configure:
   * **Name:** Enter a three-level name (`<catalog>.<schema>.<name>`). Select catalog and schema from dropdown and enter name into the textbox.
   * **Primary key:** Select the unique ID column.
   * **Columns to sync:** (Standard endpoints only) Choose columns to include, or leave blank to sync all.
   * **Embedding source:** Choose to compute embeddings (select text column and model) or use an existing embedding column.
   * **Sync computed embeddings:** Toggle to save generated embeddings to a table.
   * **Vector search endpoint:** Select your endpoint.
   * **Sync mode:** Choose *Continuous* (auto sync) or *Triggered* (manual sync). Storage-optimized endpoints support only *Triggered*.
5. Click **Create** and monitor index creation progress.

## D. Search Methods: Query, Hybrid, and Full-Text

With the index in place, we can now perform searches using different methods:

- **Query Search:** Uses embeddings to find semantically similar chunks.
- **Hybrid Search:** Combines semantic and keyword-based search for improved relevance.
- **Full-Text Search:** Retrieves chunks based on exact keyword matches.

We’ll also demonstrate how to filter results by the `path` field to target specific documents.

In [0]:
# Get the vector search index for performing searches
index = vsc.get_index(index_name=index_name)
print(index.describe())

{'name': 'dbacademy.labuser13859245_1773140998.docs_chunked_index', 'endpoint_name': 'vs_endpoint_1', 'primary_key': 'id', 'index_type': 'DELTA_SYNC', 'delta_sync_index_spec': {'source_table': 'dbacademy.labuser13859245_1773140998.docs_chunked', 'embedding_source_columns': [{'name': 'chunk', 'embedding_model_endpoint_name': 'databricks-gte-large-en'}], 'pipeline_type': 'TRIGGERED', 'pipeline_id': '83dccdc8-5591-4443-92ae-bb6f10f2e62f'}, 'status': {'detailed_state': 'ONLINE_TRIGGERED_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://dbc-e48c1d6b-f764.cloud.databricks.com/explore/data/dbacademy/labuser13859245_1773140998/docs_chunked_index', 'indexed_row_count': 18, 'triggered_update_status': {'last_processed_commit_version': 1, 'last_processed_commit_timestamp': '2026-03-10T11:21:46Z', 'triggered_update_progress': {'latest_version_currently_processing': 1, 'num_synced_rows': 18, 'total_rows_to_sync': 18, 'sync_progress_completion': 1.0, 'estimated_completion_ti

### D1. Query Search: Similarity Search

In this step, we'll use the vector search index to perform a semantic search. This method retrieves document chunks that are contextually similar to our query, even if the exact keywords don't match. We'll use this approach to find relevant information based on meaning, not just keywords.

In [0]:
query_text = "How does the Orion system prevent overheating during continuous operation?"
results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=3
)
display(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 3,
  'data_array': [['dbfs:/Volumes/dbacademy/labuser13859245_1773140998/orion_docs/02_Orion_Battery_Management_System_Manual.pdf',
    "Orion Robotics\n\nOrion A1 - Battery Management System Manual\n© 2025 Orion Robotics – Internal Engineering Reference\nThis manual provides an in-depth overview of the Orion A1 Battery Management System (BMS), covering hardware design, safety mechanisms, and maintenance best practices. The BMS is a critical subsystem responsible for monitoring, protecting, and optimizing the lithium-ion battery modules powering the Orion A1 platform.\nThe Orion BMS ensures long-term battery health by dynamically balancing cell voltages, regulating current flow, and monitoring thermal conditions through embedded temperature sensors. The system's adaptive charge algorithm was developed in collaboration with the Orion Power Systems division, a

### D2. Hybrid Search: Semantic + Keyword

Hybrid search combines semantic similarity with keyword matching. This approach is useful when we want to balance contextual relevance with exact keyword hits, improving the precision of our search results.

In [0]:
query_text = "Explain safety verification under ISO 13849-1."
results_hybrid = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    query_type="hybrid",
    num_results=5
)
display(results_hybrid)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 5,
  'data_array': [['dbfs:/Volumes/dbacademy/labuser13859245_1773140998/orion_docs/06_Orion_Safety_and_Compliance_Sheet_v2.docx',
    "== page ==\n\nSafety and Compliance Sheet – Orion Robotics\n\n4. Software and Firmware Compliance\n\nFirmware integrity and traceability form the foundation of Orion's safety certification. Each firmware release undergoes digital signature verification using SHA-256 cryptographic hashing.\nVersion control procedures comply with IEC 61508 and internal standard OSS-102. Change management includes peer review, automated regression testing, and cross-signing by the Orion Firmware Authority (OFA).\n<table><tr><th>Clause</th><th>Requirement</th><th>Verification</th><th>Status</th></tr><tr><td>ISO 13849-1 §4</td><td>Functional safety validation</td><td>Automated test harness</td><td>Compliant</td></tr><tr><td>IEC 61508 §7</td><td>S

The model embedding may focus semantically on safety and verification but miss the specific standard reference if embeddings were trained mostly on general English text.

Keyword filtering on **"ISO 13849-1"** ensures only chunks mentioning the compliance standard are retrieved. Note that the first result includes **"ISO 13849-1"** while others don't.

### D3. Full-Text Search: Keyword Only

Full-text search retrieves document chunks based exclusively on exact keyword matches. Use this method when you require highly precise, literal results for your search terms.

**🚨 Important:** Full-text search is **currently in beta** and must be enabled for your workspace before use. You can **unskip** the code cell below after enabling this feature. You can check [this documentation page](https://docs.databricks.com/aws/en/admin/workspace-settings/manage-previews#-manage-account-level-previews) for instructions to enable a **preview** feature.

In [0]:
%skip
query_text = "PID coefficients"
results_fulltext = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    query_type="FULL_TEXT",
    num_results=5
)
display(results_fulltext)


### D4. Filter by Path: Target Specific Documents

We can use filtering to return results for specific documents. For example, we can filter search results by the `path` field to target only certain documents. This is useful when we want to restrict our search to a particular file or document within our dataset.

We'll use a `LIKE` filter here. Please note that filtering matches whitespace-separated tokens in a string. You can check the [documentation](https://docs.databricks.com/aws/en/vector-search/query-vector-search) for other supported filters.

In [0]:
query_text = "How does the Orion system prevent overheating during continuous operation?"
filtered_results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    filters={"path LIKE" : "dbfs:/Volumes/main/default/documents/03_Orion_Motion_Controller_Firmware_Guide_v6.pdf"}, # TODO: Change the path if you used a different catalog and schema
    num_results=3
)

display(filtered_results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 0}}

**💡 Additional Filter Examples:**

You can use various filter types with vector search:

```python
# Filter by exact path match
filters={"path NOT": "specific/document/path.pdf"}

# Filter by numeric ranges (if you have numeric metadata)
filters={"page_number >": 10, "page_number <": 50}
```

## E. Improving Precision through Re-Ranking

While embeddings are powerful for finding semantically similar content, they may sometimes return results that are close in meaning but weak in context. Re-ranking helps boost precision by re-evaluating the top results using a more context-aware model or additional signals.

**Why re-rank?**
- Embedding-based search can surface semantically similar but contextually irrelevant chunks.
- Re-ranking applies a secondary scoring step, often using a cross-encoder or LLM, to prioritize the most relevant results.
- In Databricks, you can use the built-in reranker to re-score and reorder the top N results from a similarity search, leveraging deeper contextual understanding.
- This improves the quality of the final output, especially for nuanced queries and critical use cases.

**Trade-off:** Re-ranking increases computational cost but can significantly improve accuracy for high-value queries. Use reranking selectively to balance performance and precision.

In [0]:
# Example: Re-rank top N results from a semantic search using DatabricksReranker

from databricks.vector_search.reranker import DatabricksReranker

query_text = "How does the Orion system prevent overheating during continuous operation?"
results_reranked = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=5,
    reranker=DatabricksReranker(columns_to_rerank=["chunk"])
)

display(results_reranked)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 5,
  'data_array': [['dbfs:/Volumes/dbacademy/labuser13859245_1773140998/orion_docs/06_Orion_Safety_and_Compliance_Sheet_v2.docx',
    'Safety and Compliance Sheet – Orion Robotics\n\nOrion A1 - Safety and Compliance Sheet\n\n1. Introduction and Scope\n\nThis document defines the mandatory safety and compliance standards governing the Orion A1 humanoid robotic platform.\nIt ensures consistent adherence to international and regional safety directives while guiding field engineers, integrators, and compliance officers through procedural safeguards.\nEach Orion system operates under a certified compliance lifecycle aligned with ISO 10218-1, ISO 13849-1, IEC 61500, and CE Machinery Directive 2006/42/EC. Each modification to hardware or firmware requires validation through the Orion Safety Verification Framework (OSVF).\n2. General Safety Principles\n\nSafety des

**💡 Question:** Compare these results with the results returned by similarity search at the beginning of this section. Do you see any improvements?

## F. Best Practices for Vector Search

1. **Minimize embedding dimensionality when possible**
   Higher-dimensional embeddings (e.g., 1024-1536) may capture more nuance but increase latency and reduce throughput. Choose the lowest dimension that preserves retrieval quality.
   *Example:* If we test a 768-dim model and a 384-dim model and find similar retrieval accuracy, we should prefer the 384-dim variant for faster queries.

2. **Keep `num_results` per query moderate (e.g., 10-100)**
   Requesting too many results increases scanning and latency; the documentation recommends staying in this range unless our use case justifies it.
   *Example:* Use `num_results=50` by default rather than `num_results=5000`.

3. **Select the correct endpoint SKU and size your index appropriately**
   Choose between "Standard" and "Storage-Optimized" endpoints based on vector count, dimension, query latency, and cost. Also ensure your index size stays within the capability of a vector search unit for optimal latency.
   *Example:* For <2 million vectors at 768 dimensions, use the Standard SKU; for >10 million vectors, consider Storage-Optimized.

4. **Use filters and metadata to narrow retrieval scope**
   Attaching metadata (e.g., document type, path prefix, source) allows us to restrict searches via `filters` and avoid retrieving irrelevant chunks, improving relevance and performance.
   *Example:* Filter by `"document_type":"manual"` so only manuals are searched for "maintenance interval" queries.

5. **Prefer ANN retrieval for speed; use hybrid (vector + keyword) when domain keywords matter**
   Approximate Nearest Neighbor (ANN) retrieval gives highest QPS and lowest latency; hybrid search should only be used when keyword relevance (e.g., legal standard codes) is critical.
   *Example:* Use ANN for general "how to recalibrate sensor" queries; use hybrid when the query must match a specific standard like "ISO 13849-1".

For more best practices, see the [Databricks Vector Search documentation](https://docs.databricks.com/aws/en/generative-ai/vector-search-best-practices).

## G. Summary

In this demo, we explored the end-to-end process of building a vector search solution for document retrieval on Databricks:

* Computed semantic embeddings for document chunks using the GTE model from Foundation Model APIs.
* Created a vector search index using both SDK and UI methods, leveraging a pre-created endpoint.
* Demonstrated query, hybrid, and full-text search methods, including filtering by document path.
* Improved retrieval precision through re-ranking, highlighting its value when embeddings return semantically close but contextually weak matches.
* Reviewed best practices for balancing computational cost, accuracy, embedding dimensionality, chunking strategy, and index refresh modes.

By following these steps and best practices, we can implement robust, high-precision retrieval systems that scale with our data and business needs.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>